# 3. Path Analysis — How Many Jumps?

**Goal**: Analyze fund flow paths from attacker wallets to determine:
1. **Optimal expansion depth** — How many hops before we stop finding useful info?
2. **Direction limits** — When does tracing incoming vs outgoing matter?
3. **Terminal detection** — At which hop do funds hit exchanges/mixers?
4. **Value decay** — How quickly does tracked value diminish per hop?

This notebook uses the **pure algorithm** from `api.algorithms.path_analyzer` — no DB access in the algorithm itself.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from notebooks.src.data_loader import DataLoader
from notebooks.src.investigation_visualizer import InvestigationVisualizer as iviz
from api.algorithms.path_analyzer import (
    build_adjacency, find_shortest_paths, find_all_paths,
    compute_path_stats, compute_jump_levels
)

loader = DataLoader()
print('Connected')

## 3.1 Load Investigation Graph

In [ ]:
INVESTIGATION_ID = 1  # Change as needed

transfers = loader.get_transfers(INVESTIGATION_ID)
wallets = loader.get_wallets(INVESTIGATION_ID)
edges_df = loader.get_edges(INVESTIGATION_ID)

print(f'Transfers: {len(transfers)}')
print(f'Wallets: {len(wallets)}')
print(f'Edges (aggregated): {len(edges_df)}')

# Find seed addresses
seed_wallets = wallets[wallets['depth'] == 0]
attacker_wallets = wallets[wallets['role'].isin(['attacker', 'theft_origin'])]
print(f'Seed wallets: {len(seed_wallets)}')
print(f'Attacker wallets: {len(attacker_wallets)}')

## 3.2 Path Statistics

In [ ]:
# Convert DataFrames to dicts for algorithm input
wallets_list = wallets.to_dict('records') if not wallets.empty else []
edges_list = edges_df.to_dict('records') if not edges_df.empty else []
attacker_addrs = attacker_wallets['address'].tolist() if not attacker_wallets.empty else []

# Compute path statistics
stats = compute_path_stats(wallets_list, edges_list, attacker_addrs)

print(f"Max depth: {stats['max_depth']}")
print(f"Avg depth: {stats['avg_depth']:.1f}")
print(f"Terminal wallets: {stats['terminal_wallets']} / {stats['total_wallets']}")
print(f"\nDepth distribution:")
for d, count in sorted(stats['depth_distribution'].items()):
    print(f"  Depth {d}: {count} wallets")

In [ ]:
# Visualize role composition at each depth
role_by_depth = stats.get('role_by_depth', {})
if role_by_depth:
    depths = sorted(role_by_depth.keys())
    all_roles = set()
    for roles in role_by_depth.values():
        all_roles.update(roles.keys())
    
    fig = go.Figure()
    for role in sorted(all_roles):
        counts = [role_by_depth.get(d, {}).get(role, 0) for d in depths]
        fig.add_trace(go.Bar(
            x=[f'Depth {d}' for d in depths],
            y=counts,
            name=role,
            marker_color=iviz.ROLE_COLORS.get(role, '#c7c7c7'),
        ))
    
    fig.update_layout(
        barmode='stack', title='Wallet Roles by Depth Level',
        xaxis_title='Depth', yaxis_title='# Wallets',
        template='plotly_white', height=450,
    )
    fig.show()

## 3.3 Value Decay Per Hop

In [ ]:
# How much value passes through each depth level?
value_by_depth = stats.get('value_by_depth', {})
if value_by_depth:
    depths = sorted(value_by_depth.keys())
    values = [value_by_depth[d] for d in depths]
    
    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=[f'Depth {d}' for d in depths],
        y=values,
        marker_color='#1f77b4',
    ))
    fig.add_trace(go.Scatter(
        x=[f'Depth {d}' for d in depths],
        y=values,
        mode='lines+markers',
        line=dict(color='#d62728', width=2),
        name='Trend',
    ))
    fig.update_layout(
        title='Total Value by Depth Level',
        xaxis_title='Depth', yaxis_title='Total Value (sum of received + sent)',
        template='plotly_white', height=400,
    )
    fig.show()
    
    # Compute decay rate
    if len(values) > 1 and values[0] > 0:
        decay_rates = [values[i] / values[i-1] if values[i-1] > 0 else 0
                       for i in range(1, len(values))]
        print(f'\nValue decay rate per hop: {[f"{r:.1%}" for r in decay_rates]}')
        print(f'Average decay: {np.mean(decay_rates):.1%}')

## 3.4 Shortest Paths to Exchanges/Mixers

In [ ]:
# Find how many hops from seed to each exchange/mixer endpoint
forward_adj, reverse_adj = build_adjacency(edges_list)

terminal_wallets = wallets[wallets['role'].isin(['exchange', 'mixer', 'bridge'])]

for _, attacker in attacker_wallets.iterrows():
    paths = find_shortest_paths(forward_adj, attacker['address'].lower(), max_depth=10)
    
    print(f"\nFrom attacker {attacker['address'][:12]}...")
    for _, terminal in terminal_wallets.iterrows():
        t_addr = terminal['address'].lower()
        if t_addr in paths:
            path = paths[t_addr]
            print(f"  → {terminal['role']} {t_addr[:12]}... = {len(path)-1} hops")
            # Show path
            path_str = ' → '.join([f"{a[:8]}" for a in path])
            print(f"    Path: {path_str}")

## 3.5 Optimal Depth Recommendation

In [ ]:
# Analyze at which depth we've captured most of the "interesting" wallets
role_by_depth = stats.get('role_by_depth', {})

interesting_roles = {'exchange', 'mixer', 'bridge', 'suspect'}
interesting_by_depth = {}
cumulative = 0

for d in sorted(role_by_depth.keys()):
    interesting_count = sum(role_by_depth[d].get(r, 0) for r in interesting_roles)
    cumulative += interesting_count
    interesting_by_depth[d] = cumulative

if interesting_by_depth:
    total_interesting = max(interesting_by_depth.values())
    for d, cum in interesting_by_depth.items():
        pct = cum / total_interesting * 100 if total_interesting > 0 else 0
        print(f"Depth {d}: {cum}/{total_interesting} interesting wallets ({pct:.0f}%)")
        if pct >= 90:
            print(f"\n→ RECOMMENDATION: max_depth={d} captures {pct:.0f}% of interesting wallets")
            break
else:
    print('No investigation data available yet.')
    print('Run an investigation first with: POST /investigations/{id}/investigate')

## 3.6 Fund Flow Timeline (Grouped by Jump Level)

In [ ]:
# Main timeline visualization
if not transfers.empty and not wallets.empty:
    fig = iviz.plot_fund_flow_timeline(
        transfers, wallets,
        title=f'Investigation {INVESTIGATION_ID} — Fund Flow Timeline'
    )
    fig.show()
else:
    print('Need transfer + wallet data to build timeline.')

## 3.7 Limitations

| Limitation | Impact | Tag |
|:-----------|:-------|:----|
| Single-chain only | Can't trace through bridges | `aria_cross_chain_manual` |
| No contract call analysis | Missing internal transactions | `needs_trace_api` |
| Linear depth ≠ actual hops | Two wallets at depth 2 may have different real distances | Use `compute_jump_levels` for BFS distance |
| Performance at scale | >10K wallets = slow graph ops | Pre-compute adjacency, use NetworkX for large graphs |